# The Reform over the Business Cycle: Steady-State Analysis with Endogenous Job Destruction

This notebook presents a search-and-matching model with endogenous job destruction and aggregate productivity shocks, following Mortensen and Pissarides (1994). The model is extended with time-limited unemployment benefits and used to analyze the effects of increasing the benefit expiration rate across macroeconomic states.

---

## Table of Contents
1. [Model Environment](#1)
2. [Steady State Equilibrium](#2)
3. [Beveridge Curve and Business Cycle Asymmetries](#3)
4. [Calibration and Baseline Economy](#4)
5. [Policy Experiment](#5)

---
<a id='1'></a>
## 1. Model Environment

The model closely follows the baseline model, with three main distinctions: no endogenous search effort; endogenous job destruction through a job-specific productivity component; and a three-state macroeconomic environment.

### 1.1 Jobs Differentiation and Endogenous Job Destruction

In the previous model, exogenous shocks destroyed unprofitable jobs at an arrival rate $\mu$, forcing some jobs to transition from a productivity $y = 1$ to $y = 0$. To allow for endogenous job destruction, job productivity is defined as $p_i + \sigma y$, where $p_i$ represents a common productivity component shared across all jobs that varies with the macroeconomic state $i \in \{\text{boom, intermediate, recession}\}$, and $\sigma y$ captures a job-specific productivity component. $y$ follows $G(y)$, and without loss of generality, $y \sim U[y_{\min}, \bar{y}]$, while $\sigma$ captures the dispersion of idiosyncratic productivity.

Idiosyncratic productivity shocks arrive at rate $\mu$, changing a job's productivity from $y$ to $y'$. The productivity draw following a shock is independent of the previous productivity level and is irreversible. Matches whose productivity remains above a reservation productivity threshold survive ($J(y') \geq 0$), whereas those falling below it are endogenously destroyed. This job destruction rule satisfies the reservation property defined by $J(R) = 0$, where $R$ denotes the reservation productivity. At job creation, firms' profit maximization requires that all new jobs are created at the maximal productivity level $\bar{y}$.

| Job Productivity $y_i$ | Firm Surplus $J(y_i)$ | Worker Value $V_e(y_i)$ | Action |
|---|---|---|---|
| $y_i < R_i$ | $J(y_i) < 0$ | $V_e(y_i) < V_{Ii}$ | Job destroyed / worker quits |
| $y_i = R_i$ | $J(R) = 0$ | $V_e(R_i) = V_{Ii}$ | Job destroyed / worker quits |
| $y_i > R_i$ | $J(y_i) > 0$ | $V_e(y_i) > V_{Ii}$ | Job maintained; worker stays |

### 1.2 Macroeconomic States and State-Dependent Job Creation and Job Destruction

Three aggregate states of the economy are modeled: boom, intermediate and recession, characterized by productivity levels $p_1 > p_2 > p_3$, respectively. Transitions between states are captured in a transition matrix $\Omega_{ij}$. Workers and firms are forward-looking: decisions on vacancy posting, job acceptance and job destruction depend on both the current productivity state and expectations about future states. As a result, equilibrium job creation, job destruction, and consequent unemployment are state-dependent.

### 1.3 Unemployment Dynamics

Inflows to unemployment, determined by job destruction, are now endogenous to the model and state-dependent. As in the previous model, benefit expiration and the transition from insured to uninsured unemployment remain unchanged:
$$E \xrightarrow{\mu G(R_i)} I, \qquad I \xrightarrow{\lambda} U, \qquad I \xrightarrow{q(\theta_i)} E, \qquad U \xrightarrow{q(\theta_i)} E$$

| Concept | Expression | Logic |
|---|---|---|
| Steady-state unemployment | $u_i = \dfrac{\mu G(R_i)}{\mu G(R_i) + \theta_i q(\theta_i)}$ | Inflows from job destruction equal outflows from job finding |
| Steady-state insured unemployment | $u_i^I = \dfrac{\mu G(R_i)(1 - u_i)}{\theta_i q(\theta_i) + \lambda}$ | Stock of unemployed receiving benefits |
| Steady-state uninsured unemployment | $u_i^U = \dfrac{\lambda}{\theta_i q(\theta_i)} u_i^I$ | Stock of unemployed after benefit exhaustion |

### 1.4 Workers

Each value function captures the expected discounted payoff in that state, accounting for immediate income, transitions across employment states, idiosyncratic shocks, and aggregate macroeconomic transitions.

**Employment:**
$$rV_{ei}(y) = w_{ei}(y) + \mu\left[\int \max\{V_{ei}(y'), V_{Ii}\}\,dG(y') - V_{ei}(y)\right] + \sum_{j \neq i} \Omega_{ij}\left[\max\{V_{ej}(y), V_{Ii}\} - V_{ei}(y)\right]$$

**Insured unemployment:**
$$rV_{Ii} = b_I + \theta_i q(\theta_i)\left[V_{ei}(\bar{y}) - V_{Ii}\right] + \lambda(V_{Ui} - V_{Ii}) + \sum_{j \neq i} \Omega_{ij}\left[V_{Ij} - V_{Ii}\right]$$

**Uninsured unemployment:**
$$rV_{Ui} = b_U + \theta_i q(\theta_i)\left[V_{ei}(\bar{y}) - V_{Ui}\right] + \sum_{j \neq i} \Omega_{ij}\left[V_{Uj} - V_{Ui}\right]$$

### 1.5 Firms

**Filled job:**
$$rJ_{ei}(y) = p_i + \sigma y - w_i(y) + \mu\left[\int \max\{J_{ei}(y'), V_i\}\,dG(y') - J_{ei}(y)\right] + \sum_{j \neq i} \Omega_{ij}\left[\max\{J_{ej}(y), V_j\} - J_{ei}(y)\right]$$

**Vacancy:**
$$rV_{vi} = -\gamma + q(\theta_i)\left[J_{ei}(\bar{y}) - V_i\right] + \sum_{j \neq i} \Omega_{ij}\left[V_{vj} - V_{vi}\right]$$

**Free entry:** $V_{vi} = 0$

In [ ]:
% file: Toolbox/qtheta.m
function q = qtheta(theta, mu, A)
% q(theta) = A * theta^(-mu)
q = A .* (theta .^ (-mu));
end

In [ ]:
% file: Toolbox/thetaqtheta.m
% theta * q(theta) = f(theta) = scale * theta^(1-eta)
% This is the job-finding rate for unemployed workers (worker-side contact rate)
function jc = thetaqtheta(theta, eta, scale)
jc = scale .* (theta .^ (1 - eta));
end

---
<a id='2'></a>
## 2. Steady State Equilibrium

The model is solved by expressing equilibrium conditions in terms of the match surplus, which measures the value of a worker–firm match relative to their respective outside options. Two central equilibrium conditions are derived:
1. **Job creation condition**: obtained from the free-entry condition and the surplus sharing rule, determines firms' incentives to post vacancies and pins down labor market tightness $\theta$.
2. **Job destruction condition**: obtained from the reservation property and surplus sharing rule, identifies reservation productivity $R_i$ below which matches are endogenously terminated.

### 2.1 Match Surplus and Wage

The match surplus is defined as:
$$S_{ei}(y) = V_{ei}(y) - V_{Ii} + J_{ei}(y) - J_{Vi}$$

Wages are determined by Nash bargaining:
$$w_{ei}(y) = \arg\max\,[V_{ei}(y) - V_{Ii}]^\beta [J_{ei}(y) - J_{Vi}]^{1-\beta}$$

with sharing rules $V_{ei}(y) - V_{Ii} = \beta S_{ei}(y)$ and $J_{ei}(y) - J_{Vi} = (1-\beta)S_{ei}(y)$.

Using the free-entry condition, Bellman equations, and the sharing rule, the surplus expression as a function of model parameters is:

$$\left(r + \mu + \sum_{j \neq i} \Omega_{ij}\right) S_{ei}(y) = p_i + \sigma y - \underbrace{\left[b_I + \theta_i \frac{\beta}{1-\beta}\gamma + \frac{\lambda}{r + \lambda + \theta_i q(\theta_i)}(b_U - b_I)\right]}_{B_i} + \mu \mathbb{E}[S_{ei}] + \sum_{j \neq i} \Omega_{ij} \max[S_{ej}(y), 0] \tag{5}$$

where $\mathbb{E}[S_{ei}] = \int \max[S_{ei}(y'), 0]\,dG(y')$.

The effective outside option $B_i$ is:
$$B_i = b_I + \theta_i \frac{\beta}{1-\beta}\gamma + \frac{\lambda}{r + \lambda + \theta_i q(\theta_i)}(b_U - b_I)$$

The wage equation reflects Nash bargaining between the worker and the firm:
$$w_i(y) = (1-\beta)\,B_i + \beta(p_i + \sigma y)$$

### 2.2 Job Creation

The job creation condition is pinned down by the free-entry condition and the wage bargaining sharing rule. It dictates that firms in a specific economic state $i$ will continue to post vacancies until the marginal cost of recruitment equals the firm's expected share of the match surplus:

$$\left(r + \mu + \sum_{j \neq i} \Omega_{ij}\right) \frac{\gamma}{q(\theta_i)} = (1-\beta)\left[p_i + \sigma\bar{y} + \mu\mathbb{E}(S_{ei}) - B_i\right] + \sum_{j \neq i} \Omega_{ij}\frac{\gamma}{q(\theta_j)} \tag{6}$$

The left-hand side represents the capitalized cost of hiring. The right-hand side represents the firm's share of surplus from a match and the expected capital gain or loss resulting from a transition between aggregate states. From the job creation equation, $\theta_1 > \theta_2 > \theta_3$: higher productivity levels imply higher incentives for firms to post new vacancies.

### 2.3 Job Destruction

Job destruction is pinned down from $S_{ei}(R_i) = 0$:

$$p_i + \sigma R_i = B_i - \mu\mathbb{E}(S_{ei}) - \sum_{j \neq i} \Omega_{ij} \max[S_{ej}(R_i), 0] \tag{7}$$

For a match to survive, its productivity must be high enough to cover the net cost of keeping the match alive. The idiosyncratic option value $\mu\mathbb{E}(S_{ei})$ captures labour hoarding: even if the match is losing money today, the firm keeps the worker because there is a chance the worker's individual productivity will reset to a high level. From the job destruction equation, $R_1 < R_2 < R_3$: a higher $p_i$ implies a lower reservation productivity.

The macroeconomic option value $\sum \Omega_{ij} \max[S_{ej}(R_i), 0]$ reflects the capital gain or loss accruing to the match surplus from aggregate state transitions. Only transitions toward states in which the surplus remains non-negative contribute to survival:

| State | Effective Term | Interpretation |
|---|---|---|
| Boom (1) | $0$ | Match at $R_1$ is destroyed in states 2 and 3 |
| Intermediate (2) | $\Omega_{23}\max[S_{e3}(R_2), 0]$ | Match at $R_2$ is destroyed only in recession |
| Recession (3) | $\Omega_{31}\max[S_{e1}(R_3), 0] + \Omega_{32}\max[S_{e2}(R_3), 0]$ | Match at $R_3$ survives in states 1 and 2 |

### 2.4 Expected Match Continuation Surplus

Both job creation and job destruction depend on $\mathbb{E}(S_{ei})$, the future match surplus after an idiosyncratic shock. The surplus functions $S_{ei}(y)$ are piecewise linear in $y$, with slopes $b_{ij}$ that differ across productivity regions. These slopes are obtained by solving a linear system $AS' = b$ for each region:

| Region | Linear system |
|---|---|
| $y \in [R_3, \bar{y}]$ | All three states active: $(r+\mu+\Omega_{12}+\Omega_{13})S'_1 - \Omega_{12}S'_2 - \Omega_{13}S'_3 = \sigma$, etc. |
| $y \in [R_2, R_3]$ | High and Medium active; Low: $S'_3 = 0$ |
| $y \in [R_1, R_2]$ | Only High active: $(r+\mu+\Omega_{12}+\Omega_{13})S'_1 = \sigma$; $S'_2 = S'_3 = 0$ |
| $y \in [y_{\min}, R_1]$ | $S'_1 = S'_2 = S'_3 = 0$ |

The expected surplus for each state is then:
$$\mathbb{E}[S_{ei}] = \sum_j b_{ij} \int_{R_j}^{R_{j+1}} [1 - G(y')]\,dy' \tag{F}$$

For a uniform distribution, $1 - G(y) = (\bar{y} - y)/(\bar{y} - y_{\min})$, so:
$$\int_{R_j}^{R_{j+1}} [1-G(y')]\,dy' = \int_{R_j}^{R_{j+1}} \frac{\bar{y} - y'}{\bar{y} - y_{\min}}\,dy'$$

### 2.5 Equilibrium System

We consider the three aggregate states $i = 1, 2, 3$, with $p_1 > p_2 > p_3$. The equilibrium is characterized by the following system of six equations in six unknowns $(R_1, R_2, R_3, \theta_1, \theta_2, \theta_3)$:

**Job creation conditions** (eq. 4, 5, 6 in the code):

$$\left(r+\mu+\sum_{j\neq i}\Omega_{ij}\right)\frac{\gamma}{q(\theta_i)} = (1-\beta)\left[p_i + \sigma\bar{y} + \mu\mathbb{E}(S_{ei}) - B_i\right] + \sum_{j\neq i}\Omega_{ij}\frac{\gamma}{q(\theta_j)}$$

**Job destruction conditions** (eq. 1, 2, 3 in the code):

$$p_i + \sigma R_i = B_i - \mu\mathbb{E}(S_{ei}) - \sum_{j\neq i}\Omega_{ij}\max[S_{ej}(R_i), 0]$$

---
<a id='3'></a>
## 3. Beveridge Curve and Business Cycle Asymmetries

### 3.1 Beveridge Curve

The model introduces two channels impacting the relation between vacancies and unemployment.

**The Matching Effect.** Holding the destruction rate constant, an increase in vacancies increases market tightness $\theta$, thereby increasing the probability of an unemployed worker finding a job. To maintain a stationary matching rate, the unemployment level must fall. This generates the classical downward-sloping segment of the Beveridge curve.

**The Job Destruction Channel.** As $v$ increases, the opportunity cost of employment increases, forcing firms to increase their reservation productivity threshold. The job destruction rate thus increases with $\theta$. This induces higher turnover, requiring a higher unemployment level to reach a stationary state. The slope of the Beveridge curve depends on the relative magnitudes of these effects: if the elasticity of job destruction with respect to tightness exceeds the elasticity of the matching function, the model predicts a breakdown in the standard inverse relationship between vacancies and unemployment.

### 3.2 Asymmetries in Job Creation and Job Destruction Across the Business Cycle

Firms can adjust job destruction and vacancies immediately when conditions change — they are forward-looking "jump" variables. Unemployment, however, adjusts gradually over time and is a sticky variable. This generates asymmetric dynamics:

**When productivity falls (recession):** Some jobs that were just barely profitable are no longer worth keeping. Firms immediately destroy these marginal jobs. Transition from high to low productivities induces an immediate upward shift in the reservation productivity threshold. This shift results in an instantaneous cleansing effect: the entire stock of existing jobs whose idiosyncratic productivity falls between the old and new reservation levels is terminated on impact.

**When productivity rises (boom):** Firms respond by opening new vacancies right away, since hiring is more attractive. However, these vacancies take time to fill, so employment does not increase immediately.

This structural disparity between the immediate termination of marginal jobs and the gradual matching of new ones explains why the velocity of unemployment's rise during a downturn is markedly faster than its rate of decline during a subsequent boom.

---
<a id='4'></a>
## 4. Calibration and Baseline Economy

The model is calibrated to quarterly values to simulate an artificial economy using parameters from the literature.

| Parameter | Symbol | Value | Source |
|---|---|---|---|
| Quarterly interest rate | $r$ | 0.0125 | Standard |
| Bargaining power | $\beta$ | 0.500 | Hosios (1990) |
| Matching elasticity | $\eta$ | 0.500 | Petrongolo and Pissarides (2001) |
| Benefit expiration hazard | $\lambda$ | 0.1875 | Unédic (2022), 16-month avg. |
| Insured UI benefit | $b_I$ | 0.55 | OECD replacement rate |
| Uninsured income | $b_U$ | 0.15 | RSA + transfers, CAF (2022) |
| Vacancy posting cost | $\gamma$ | 0.40 | Hagedorn and Manovskii (2008) |
| Matching efficiency | $m_0$ | 1 | Petrongolo and Pissarides (2001) |
| Idiosyncratic shock dispersion | $\sigma$ | 0.3637 | L'Haridon and Malherbet (2009) |
| Idiosyncratic shock hazard rate | $\mu$ | 0.08 | L'Haridon and Malherbet (2009) |

The aggregate technology shock is modeled as a three-state Markov chain on the grid $(p_1, p_2, p_3)$, with transition probabilities $\Omega_{ij}$, designed to approximate an underlying AR(1) process with $\rho = 0.94$ and $\nu = 0.007$. The idiosyncratic component $y$ is assumed to follow a uniform distribution on $[0, 2]$.

In [ ]:
clearvars; close all;
addpath('Toolbox')



%% SECTION 1: Environment / Initialization / Parameters

%-----% Parameters / env. %-----------------------------------------------%

kpic                = 1;

r_annual            = 0.05;
r_quarterly        =  (1+r_annual)^(1/4)-1;
%r_monthly          = (1+r_annual)^(1/12)-1;
%r_daily            = (1+r_annual)^(1/360)-1;
r                   = r_quarterly;

% AR(1) process  
rho                 = 0.94 ;                                                 % Autocorrelation coefficient
nu                  = 0.007;                                                 % Std deviation of the innovation

var_nu              = (nu^2)/(1-rho^2);
m                   = 3;   
p1                  = +m*(var_nu)^0.5;                                      % HS: high state
p2                  = 0;                                                    % MS: med state
p3                  = -m*(var_nu)^0.5;                                   % LS: low state
pvec                = [p1, p2 p3];

% Transition Probability Matrix (Omega)
Ti                  = zeros(3,3);
Ti(:,:)             = [ rho             1-rho           0
                        (1-rho)/4       (1+rho)/2       (1-rho)/4
                        0               1-rho           rho
                        ];



% Poisson process
ymin                = 0;
ymax                = 2;
mu                  = 0.08;                  
sigma               = 0.3637; 
lambda              = 0.1875;

% Flow cost
gamma               =0.4;

% Matching
eta                 = 0.5;
scale               = 1;

% Bargaining
beta                = 0.5;

% Policy parameters
par.bi = 0.55;  
par.bu = 0.15;
% Evaluate: PDM for expectation terms
%
% 

% y in [R3 , ymax]
Ai      = zeros(3,3);																					 
Ai(1,1) = r+mu+Ti(1,2)+Ti(1,3);         Ai(1,2) = -Ti(1,2);                         Ai(1,3) = -Ti(1,3);                 % HS
Ai(2,1) = -Ti(2,1);                         Ai(2,2) = r+mu+Ti(2,1)+Ti(2,3);         Ai(2,3) = -Ti(2,3);                 % MS
Ai(3,1) = -Ti(3,1);			        Ai(3,2) = -Ti(3,2);			        Ai(3,3) = r+mu+Ti(3,1)+Ti(3,2); % LS
Gi      = [sigma sigma sigma]';
V1      = Ai\Gi;

% y in [R2, R3]
Bi      = zeros(3,3);
Bi(1,1) = r+mu+Ti(1,2)+Ti(1,3);         Bi(1,2) = -Ti(1,2);                         Bi(1,3) = 0;                        % HS
Bi(2,1) = -Ti(2,1);                         Bi(2,2) = r+mu+Ti(2,1)+Ti(2,3);         Bi(2,3) = 0;                        % MS
Bi(3,1) = 0;                                Bi(3,2) = 0;                                Bi(3,3) = r+mu+Ti(3,1)+Ti(3,2); % LS
Gi      = [sigma sigma 0]';
V2      = Bi\Gi;

% y in [R1, R2]
Ci      = zeros(3,3);
Ci(1,1) = r+mu+Ti(1,2)+Ti(1,3);         Ci(1,2) = 0;                                Ci(1,3) = 0;                        % HS
Ci(2,1) = 0;                                Ci(2,2) = r+mu+Ti(2,1)+Ti(2,3);         Ci(2,3) = 0;                        % MS
Ci(3,1) = 0;                                Ci(3,2) = 0;                                Ci(3,3) = r+mu+Ti(3,1)+Ti(3,2); % LS
Gi      = [sigma 0 0]';
V3      = Ci\Gi;

% y in [ymin, R1]
Di      = zeros(3,3);
Di(1,1) = r+mu+Ti(1,2)+Ti(1,3);         Di(1,2) = 0;                                Di(1,3) = 0;                        % HS
Di(2,1) = 0;                                Di(2,2) = r+mu+Ti(2,1)+Ti(2,3);         Di(2,3) = 0;                        % MS
Di(3,1) = 0;                                Di(3,2) = 0;                                Di(3,3) = r+mu+Ti(3,1)+Ti(3,2); % LS
Gi      = [0 0 0]';
V4      = Di\Gi;

% Partial derivatives matrix (PDM)
pdm                 = [V1 V2 V3 V4];

% Evaluate: Ergodic prob.
[V, D]              = eig(Ti');                                             % Compute eigenvalues and eigenvectors
eig_val             = diag(D);
ergodic_idx         = find(abs(eig_val - 1) < 1e-10);
ergodic_vector      = V(:,ergodic_idx)';
ergodic_probs       = ergodic_vector / sum(ergodic_vector);                 % Normalize ergodic probabilities
ergpv               = ergodic_probs;


% Create structure for parameters
par.rho               = rho;

par.ymin            = ymin;
par.ymax            = ymax;
par.mu              = mu;
par.sigma           = sigma;

par.gamma           = gamma;
par.eta             = eta;
par.scale           = scale;
par.lambda          =lambda;
par.r               = r;

par.beta            = beta;
par.bi           = par.bi;  
par.bu           = par.bu;

par.pdm             = pdm;
par.ergpv           = ergpv;
par.pvec            = pvec;
par.Ti              = Ti;

**Note on the PDM (Partial Derivatives Matrix).** The matrices `Ai`, `Bi`, `Ci`, `Di` correspond to the four productivity regions $[R_3, \bar{y}]$, $[R_2, R_3]$, $[R_1, R_2]$, $[y_{\min}, R_1]$ respectively. Each system $AS' = b$ gives the slope coefficients $b_{ij}$ of the piecewise linear surplus functions in each region (Appendix F). The columns of `pdm = [V1 V2 V3 V4]` store these slopes, which are later used in `equilibrium.m` to compute $\mathbb{E}[S_{ei}]$ via numerical integration.

In [ ]:
%% SECTION 2: Equilibrium with aggregate shocks
%
%

x0                  = [1.6143    1.6796    1.7520    0.5423    0.3788    0.2283]; % R1 R2 R3 theta1 theta2 theta3
options             = optimset('display','iter','Algorithm','trust-region-dogleg',...
    'Diagnostics','off','MaxIter',1000,'Tolfun',1e-8,'TolX',1e-8); 
[x,FVAL,EXITFLAG]   = fsolve(@(x) equilibrium(x,par), x0, options);


EXITFLAG
xstar               = x;

% Compute aggregate unemployment rate
R1                  = x(1);
R2                  = x(2);
R3                  = x(3);
ts1                 = x(4);
ts2                 = x(5);
ts3                 = x(6);


fun_rep             = @(y) (y-ymin)/(ymax-ymin);
jc1                 = thetaqtheta(ts1, eta, scale);
jc2                 = thetaqtheta(ts2, eta, scale);
jc3                 = thetaqtheta(ts3, eta, scale);
jd1                 = mu*fun_rep(R1);
jd2                 = mu*fun_rep(R2);
jd3                 = mu*fun_rep(R3);
u1                  = jd1/(jd1+jc1);
u2                  = jd2/(jd2+jc2);
u3                  = jd3/(jd3+jc3);
ui1                 = (jd1 * (1-u1)) / (jc1 + lambda); 
ui2                 = (jd2 * (1-u2)) / (jc2 + lambda); 
ui3                 = (jd3 * (1-u3)) / (jc3 + lambda);
uu1                 = lambda * ui1 / jc1 ; 
uu2                 = lambda * ui2 / jc2 ; 
uu3                 = lambda * ui3 / jc3 ; 

uvec                = [u1 u2 u3];
uivec                = [ui1 ui2 ui3];
uuvec                = [uu1 uu2 uu3];

ergpv               = par.ergpv;
uagg                = ergpv*uvec';
thetavec            = [ts1 ts2 ts3];
vacrate = ergpv * (thetavec .* uvec)';

jdvec            = [jd1 jd2 jd3];
jcvec=[jc1 jc2 jc3];
jc=ergpv*jcvec';
jd=ergpv*jdvec';

seprate = ergpv * jdvec';

fprintf('Aggregate Unemployment Rate: %4.4f\n', uagg);
fprintf('Creation: %4.4f\n', jc);
fprintf('Destruction: %4.4f\n', jd);

fprintf('--- Simulation Results ---\n');
fprintf('State:      High (1)   Medium (2)   Low (3)\n');
fprintf('R (Res.):   %4.4f     %4.4f       %4.4f\n', xstar(1:3));
fprintf('Theta:      %4.4f     %4.4f       %4.4f\n', xstar(4:6));
fprintf('--------------------------\n');
fprintf('Aggregate Unemployment: %4.4f%%\n', uagg * 100);
fprintf('Vacancy Creation Rate: %4.4f%%\n', vacrate* 100);
fprintf('Separation Rate: %4.4f%%\n', seprate* 100);

% Calculate state-specific rates using your matching function parameters
% Job finding rate f(theta) = scale * theta^(1-eta)
f1 = scale * ts1^(1-eta);
f2 = scale * ts2^(1-eta);
f3 = scale * ts3^(1-eta);

% Vacancy filling rate q(theta) = scale * theta^(-eta)
q1 = scale * ts1^(-eta);
q2 = scale * ts2^(-eta);
q3 = scale * ts3^(-eta);

% Calculate Aggregates (using ergodic probabilities)
f_agg = ergpv * [f1; f2; f3];
q_agg = ergpv * [q1; q2; q3];

% Print to Console
fprintf('\n--- Matching Rates ---\n');
fprintf('Aggregate Job Finding Rate (f):    %4.4f\n', f_agg);
fprintf('Aggregate Vacancy Filling Rate (q): %4.4f\n', q_agg);
fprintf('--------------------------------------\n');


% Collect state-specific results into a table
States = {'High (1)'; 'Medium (2)'; 'Low (3)'};
JC_Rates = [jc1; jc2; jc3];
JD_Rates = [jd1; jd2; jd3];
Unemployment = uvec' * 100; 
Insured_U = uivec' * 100;
Uninsured_U = uuvec' * 100;

ResultsTable = table(States, JC_Rates, JD_Rates, Unemployment, Insured_U, Uninsured_U, ...
    'VariableNames', {'State', 'Job_Creation', 'Job_Destruction', 'Total_U_pct', 'Insured_U_pct', 'Uninsured_U_pct'});

fprintf('\n--- Steady-State Results by Aggregate State ---\n');
disp(ResultsTable);

% Compute vacancies
vvec = thetavec .* uvec;

The `equilibrium.m` function implements the six-equation system. The expected surplus $\mathbb{E}[S_{ei}]$ is computed by numerical integration using the precomputed slope coefficients from `pdm`, with `intES(x, par)` returning $1 - G(x) = (\bar{y} - x)/(\bar{y} - y_{\min})$ for the uniform distribution. The cross-surpluses $\max[S_{ej}(R_i), 0]$ are computed analytically using the piecewise structure of the surplus functions.

In [ ]:
function Fsystem = equilibrium(x,par)

% Endogeneous variables
R1                  = x(1);                                                 % Reservation productivities
R2                  = x(2);
R3                  = x(3);

% Labor Market Tightness
ts1                 = x(4);                                                 % Labor market tightness
ts2                 = x(5);
ts3                 = x(6);


% Update parameters
r                   = par.r; % Interest rate
ymax                = par.ymax; % Max and min idiosyncratic productivity
ymin                = par.ymin;
mu                  = par.mu; % Idiosyncratic productivity shock hazard rate
lambda              = par.lambda; % Benefit expiration rate
sigma               = par.sigma; % Idiosyncratic productivity dispersion
gamma               = par.gamma; % Vacancy posting cost
eta                 = par.eta; % Matching elasticity
scale               = par.scale; % Matching efficiency
beta                = par.beta; % Bargaining power
bi                  = par.bi; % Benefits insured
bu                  = par.bu; % Assistence uninsured

pdm                 = par.pdm; 

pvec                = par.pvec; % Agg. productivity vector
p1                  = pvec(1);
p2                  = pvec(2);
p3                  = pvec(3);
Ti                  = par.Ti; % Transition matrix

% Evaluate: Vacancy filling rate
vfr1                = qtheta(ts1, eta, scale);
vfr2                = qtheta(ts2, eta, scale);
vfr3                = qtheta(ts3, eta, scale);

% Evaluate: Expectation terms
itol                = 1e-6;
ai                  = pdm;
t0                  = ai(1,3)*integral(@(x) intES(x,par), R1, R2,'AbsTol',itol);
t1                  = ai(1,2)*integral(@(x) intES(x,par), R2, R3,'AbsTol',itol);
t2                  = ai(1,1)*integral(@(x) intES(x,par), R3, ymax,'AbsTol',itol);
ES1                 = t0+t1+t2;
    
t0                  = ai(2,2)*integral(@(x) intES(x,par), R2, R3,'AbsTol',itol);
t1                  = ai(2,1)*integral(@(x) intES(x,par), R3, ymax,'AbsTol',itol);
ES2                 = t0+t1;

t0                  = ai(3,1)*integral(@(x) intES(x,par), R3, ymax,'AbsTol',itol);
ES3                 = t0;

% Evaluate: expected utility of the unemployed workers
rUi1                 = bi+beta*ts1*gamma/(1-beta) + ((lambda / (r + lambda + ts1 * vfr1)) * (bu -bi));
rUi2                 = bi+beta*ts2*gamma/(1-beta) + ((lambda / (r + lambda + ts2 * vfr2)) * (bu -bi));
rUi3                 = bi+beta*ts3*gamma/(1-beta) + ((lambda / (r + lambda + ts3 * vfr3)) * (bu -bi));

% Evaluate: Cross-surpluses (max(Sei,0)) 
Se2_R1              = 0;                                                    % <0
Se3_R1              = 0;                                                    % <0
 
t0                  = r+mu+Ti(1,2)+Ti(1,3);
t1                  = p1+sigma*R2-rUi1+mu*ES1;
Se1_R2              = t1/t0;                                                % >0
Se3_R2              = 0;                                                    % <0

as1                 = r+mu+Ti(1,2)+Ti(1,3);
as2                 = r+mu+Ti(2,1)+Ti(2,3);
as3                 = r+mu+Ti(3,1)+Ti(3,2);
asg                 = as1*as2-Ti(1,2)*Ti(2,1);
t1                  = p1+sigma*R3-rUi1+mu*ES1;
t2                  = p2+sigma*R3-rUi2+mu*ES2;
Se1_R3              = (as2*t1+Ti(1,2)*t2)/asg;                              % >0
Se2_R3              = (as1*t2+Ti(2,1)*t1)/asg;                              % >0


    
% Job destruction (HS)

eq1=p1 + sigma * R1 - (bi + ts1 * (beta / (1 - beta)) * gamma + (lambda / (r + lambda + ts1 * vfr1)) * (bu - bi ) - mu*ES1 - (Ti(1,2)*Se2_R1+Ti(1,3)*Se3_R1));


% Job destruction (MS)
eq2=p2 + sigma * R2 - (bi + ts2 * (beta / (1 - beta)) * gamma + (lambda / (r + lambda + ts2 * vfr2)) * (bu - bi) - mu*ES2 - (Ti(2,1)*Se1_R2+Ti(2,3)*Se3_R2));


% Job destruction (LS)
eq3=p3 + sigma * R3 - (bi + ts3 * (beta / (1 - beta)) * gamma + (lambda / (r + lambda + ts3 * vfr3)) * (bu - bi ) - mu*ES3 - (Ti(3,1)*Se1_R3+Ti(3,2)*Se2_R3));

% Job creation (HS)
eq4=as1 * (gamma / vfr1) - (1 - beta) * (p1 + sigma * ymax + ES1- (bi + ts1 * (beta / (1 - beta)) * gamma + (lambda / (r + lambda + ts1 * vfr1)) * (bu - bi)))- Ti(1,2)*(gamma/vfr2)- Ti(1,3)*(gamma/vfr3);

% Job creation (MS)
eq5=as2 * (gamma / vfr2) - (1 - beta) * (p2 + sigma * ymax + ES2- (bi+ ts2 * (beta / (1 - beta)) * gamma + (lambda / (r + lambda + ts2 * vfr2)) * (bu - bi ))) -Ti(2,1)*(gamma/vfr1)-Ti(2,3)*(gamma/vfr3);

% Job creation (LS)
eq6=as3 * (gamma / vfr3) - (1 - beta) * (p3 + sigma * ymax + ES3- (bi + ts3 * (beta / (1 - beta)) * gamma + (lambda / (r + lambda + ts3 * vfr3)) * (bu - bi ))) -Ti(3,1)*(gamma/vfr1)-Ti(3,2)*(gamma/vfr2);

 Fsystem = [
        eq1;
        eq2;
        eq3;
        eq4;
        eq5;
        eq6
    ];

return

function temp_intES = intES(x,par)
% update parameters
ymin            = par.ymin;
ymax            = par.ymax;

t0              = (ymax-x);
pdf             = 1/(ymax-ymin);

temp_intES = t0.*pdf;
return
% EOF

---
<a id='5'></a>
## 5. Policy Experiment

### 5.1 Theoretically Ambiguous Effect of Reduced Potential Benefit Duration

An increase in the benefit expiration rate $\lambda$ has theoretically ambiguous effects on equilibrium unemployment due to the presence of competing mechanisms.

On the one hand, a higher $\lambda$ reduces the worker's outside option in wage bargaining, which lowers the reservation productivity threshold $R$ and induces firms to create more vacancies. This contributes to increasing market tightness $\theta$ and reducing unemployment through higher job-finding rates.

On the other hand, and indirectly, a higher $\theta$ increases the value of unemployment through improved job-finding prospects, potentially raising the reservation productivity and inducing higher job destruction. As a result, the net effect of an increase in $\lambda$ on unemployment depends on the relative strength of these opposing forces, and is therefore theoretically ambiguous.

### 5.2 Results

An increase in the benefit expiration rate $\lambda$ reduces the expected duration of unemployment benefits, thereby lowering workers' outside option in wage bargaining. This leads to a decrease in the reservation productivity threshold $R$, implying that lower productivity is required to keep the match alive. As a result, firms post more vacancies, increasing market tightness $\theta$ and improving job-finding prospects. Consequently, the job-finding rate $f$ increases, while the vacancy-filling rate $q$ declines due to increased competition among vacancies. Overall, these equilibrium adjustments lead to a reduction in the unemployment rate. The effect on job destruction remains limited, as the policy primarily operates through the worker's outside option rather than the productivity process itself. Finally, the composition of unemployment shifts, with a reduction in insured unemployment and a corresponding increase in uninsured unemployment, reflecting the shorter duration of benefit eligibility.

In [ ]:
%% ---  Lambda Policy Experiment --- 
clearvars; close all;
addpath('Toolbox')
run('simulation.m');  

% Define the two lambda scenarios
lambda_scenarios = [0.1875, 0.25];
scenario_names = {'Baseline','PolicyChange'};
n_scen = length(lambda_scenarios);

% Preallocate results table
results_table = table('Size',[n_scen 8], ...
    'VariableTypes', {'string','double','double','double','double','double','double','double'}, ...
    'VariableNames', {'Scenario','Unemployment','VacancyRate','Separation','ts_mean','R_mean', 'f','q'});

for i = 1:n_scen
    fprintf('\n--- Running Scenario: %s (Lambda = %.4f) ---\n', scenario_names{i}, lambda_scenarios(i));
    
    % Update lambda
    par.lambda = lambda_scenarios(i);
    
    % Solve equilibrium
    x0 = [1.6143 1.6796 1.7520 0.5423 0.3788 0.2283];
    options = optimset('display','off','Algorithm','trust-region-dogleg', ...
                       'MaxIter',1000,'TolFun',1e-8,'TolX',1e-8);
    [x,FVAL,EXITFLAG] = fsolve(@(x) equilibrium(x,par), x0, options);
    
    % Extract outcomes
    R = x(1:3);
    ts = x(4:6);
    fun_rep = @(y) (y-par.ymin)/(par.ymax-par.ymin);
    jc = arrayfun(@(t) thetaqtheta(t, par.eta, par.scale), ts);
    jd = par.mu*fun_rep(R);
    % Steady-state vectors 
    uvec   = jd ./ (jd + jc);
    uivec  = (jd .* (1 - uvec)) ./ (jc + par.lambda); % Added dot before /
    uuvec  = (par.lambda .* uivec) ./ jc;            % Added dot before /
    
    % --- CALCULATE MISSING SCALARS ---
    uagg    = ergpv * (uivec + uuvec)'; 
    seprate = ergpv * jd';
    vacrate = ergpv * (thetavec .* uvec)';
    % Job finding (f) and Vacancy filling (q)
    f_vec = par.scale .* (ts.^(1-par.eta));
    q_vec = par.scale .* (ts.^(-par.eta));
    
    f_agg = par.ergpv * f_vec';
    q_agg = par.ergpv * q_vec';

    % --- Store results ---
    results_table.Scenario(i)     = scenario_names{i};
    results_table.Unemployment(i) = uagg;
    results_table.U_insured(i)    = ergpv * uivec';
    results_table.U_uninsured(i)  = ergpv * uuvec';
    results_table.VacancyRate(i)  = vacrate;
    results_table.Separation(i)   = seprate;
    results_table.ts_mean(i)      = ergpv * ts';
    results_table.R_mean(i)       = ergpv * R';
    results_table.F_mean(i) =f_agg;
    results_table.Q_mean(i)=q_agg;
    results_table.JC(i)=ergpv*jc';
    results_table.JD(i)=ergpv*jd';

    
    fprintf('Unemployment: %.4f, Vacancy: %.4f, Separation: %.4f\n', uagg, vacrate, seprate);
end

fprintf('\n--- Dichotomic Policy Experiment Complete ---\n');
disp(results_table);